### **Module 1: Introduction (01-intro.md)**

#### **What is an LLM?**
- **LLM (Large Language Model)** = A neural network trained on massive amounts of text
- It works like your phone's autocomplete but at a much larger scale
- When you type "how are" in WhatsApp, it suggests "you" - that's a simple language model predicting the next word
- LLMs do the same thing but with billions of parameters, making it feel like talking to an intelligent being

#### **LLMs as Black Boxes**
In this course, we treat LLMs as **black boxes**:
- **Text goes in** (prompt)
- **Text comes out** (response)
- We don't look inside or host models ourselves
- We call them via API from providers like OpenAI, Google, etc.

#### **Limitations of Plain LLMs**
1. **Knowledge Cutoff**: They only know what was in their training data. If you ask about something that happened after training, they'll either don't know or **make something up**
2. **No Access to Your Data**: They can't see your documents, databases, or internal systems unless you provide that information
3. **Hallucinations**: They sometimes produce confident-sounding answers that are simply wrong

#### **What is RAG?**
**RAG = Retrieval-Augmented Generation**

- **Retrieval** = Search for relevant documents
- **Augmented** = Add those documents to the prompt
- **Generation** = LLM produces an answer based on that context

**Why RAG?**
- Instead of hoping the LLM memorized the answer, we **retrieve the right information** and hand it to the LLM
- This lets us inject knowledge the model **never saw during training**
- The LLM only sees the documents we hand it, so its answers are **grounded in our data**

---


### **Module 2: Environment Setup (02-environment.md)**

#### **Why These Tools?**

1. **uv** - A fast Python package manager (replaces pip)
   - Faster and more convenient than pip
   - Creates `pyproject.toml` for dependency management

2. **Jupyter Notebook** - Interactive coding environment
   - Perfect for experimentation and learning
   - Run code cell by cell and see results immediately

3. **Dependencies Explained**:
   - `requests` - To fetch data from the internet (download FAQ dataset)
   - `minsearch` - A simple in-memory search engine (our database)
   - `openai` - API client to talk to the LLM
   - `jupyter` - The notebook environment
   - `python-dotenv` - Load API keys from `.env` file securely

#### **Security Best Practices**
- **NEVER commit API keys to Git** - Treat them like passwords
- Store keys in `.env` file
- Add `.env` to `.gitignore`
- If a key leaks, someone can run up charges on your account

---


### **Module 3: RAG Architecture (03-rag.md)**

#### **The Problem with Plain LLMs**

When you ask a course-specific question like *"I just discovered the course. Can I join now?"*:
- The LLM gives a **generic answer** ("you can usually join" or "check the course website")
- It doesn't know about **your specific courses**, enrollment policies, or schedules
- It tries to be helpful but has **no idea** about actual enrollment status

**Why?** Because your course data is **not in the training data**.

#### **The Solution: Add Context**

When we manually paste FAQ content into the prompt:
```
Context: "Yes, but if you want to receive a certificate, you need to submit 
your project while we're still accepting submissions."
```

Now the LLM gives the **correct answer**: *"Yes, you can still join. If you want to receive a certificate, you need to submit your project while submissions are still open."*

**This is RAG!** We retrieved relevant information and augmented the generation.

#### **The Three Components of RAG**

```
┌─────────────────────────────────────────────────────────────┐
│                        RAG PIPELINE                         │
└─────────────────────────────────────────────────────────────┘

User Question
      ↓
┌─────────────┐
│   SEARCH    │ ← Component 1: Find relevant documents
│  (Database) │    from the knowledge base
└─────────────┘
      ↓
Retrieved Documents (Context)
      ↓
┌─────────────────┐
│  BUILD PROMPT   │ ← Component 2: Combine question + context
│  (Question +    │    into a proper instruction
│    Context)     │
└─────────────────┘
      ↓
Formatted Prompt
      ↓
┌─────────────┐
│     LLM     │ ← Component 3: Generate answer
│  (OpenAI/   │    based on the context
│   Gemini)   │
└─────────────┘
      ↓
   Answer
      ↓
   User
```

**Key Insight**: Each component is **independent** and **swappable**:
- Swap **minsearch** → **Elasticsearch**? Just change the search function
- Swap **OpenAI** → **Gemini**? Just change the LLM call
- Nothing else changes!

#### **The RAG Function**

```python
def rag(question):
    search_results = search(question)           # Step 1: Retrieve
    user_prompt = build_prompt(question, search_results)  # Step 2: Build prompt
    return llm(user_prompt)                     # Step 3: Generate
```

That's the **entire architecture** in 3 lines!

---


In [1]:
# ============================================================================
# IMPORTS & ENVIRONMENT SETUP
# ============================================================================

import os
from dotenv import load_dotenv
from openai import OpenAI
# import minsearch  # You'll use this later for search

# WHY: load_dotenv() reads the .env file and loads variables into os.environ
# This keeps API keys secure and out of your code
load_dotenv()

# WHY: Initialize the OpenAI client
# This creates a connection to OpenAI's API
# It automatically picks up OPENAI_API_KEY from your .env file
openai_client = OpenAI()

In [2]:
# ============================================================================
# COMPONENT 1: THE LLM (Generation)
# ============================================================================

def llm(prompt):
    """
    WHY THIS FUNCTION EXISTS:
    - Encapsulates all LLM calls in one place
    - Makes it easy to swap providers later (OpenAI → Gemini)
    - Acts as the "black box" - text in, text out
    
    PARAMETERS:
    prompt (str): The text to send to the LLM
    
    RETURNS:
    str: The LLM's response
    """
    
    # WHY: We call the OpenAI API to generate text
    # The LLM is a black box - we send a prompt, it returns a response
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",  # WHY: Use a fast, cheap model for RAG
                              # gpt-4o-mini is good balance of speed/cost/quality
        
        messages=[
            {"role": "user", "content": prompt}
            # WHY: Modern chat APIs use a message format
            # "user" role means this is the user's question/instruction
        ],
        
        temperature=0.0  
        # WHY: Temperature controls randomness
        # 0.0 = deterministic, always gives same answer for same input
        # Important for RAG - we want consistent answers based on context
        # Higher values (0.7-1.0) = more creative but less reliable
    )
    
    # WHY: Extract the actual text from the response
    # The API returns a complex object, we just want the message content
    return response.choices[0].message.content


In [3]:
# ============================================================================
# COMPONENT 2: BUILD PROMPT (Combining Question + Context)
# ============================================================================

def build_prompt(question, context):
    """
    WHY THIS FUNCTION EXISTS:
    - LLMs don't know our private data
    - We must manually inject retrieved context into the prompt
    - This function formats everything into a clear instruction
    
    PARAMETERS:
    question (str): The user's question
    context (str): Retrieved documents from search
    
    RETURNS:
    str: A formatted prompt combining instructions, context, and question
    """
    
    prompt = f"""
Your task is to answer questions from the course participants 
based on the provided context.

Use the context to find relevant information and provide accurate 
answers. If the answer is not found in the context, 
respond with "I don't know."

Question:
{question}

Context:
{context}
"""
    # WHY this structure:
    # 1. Clear instruction - tells the LLM what to do
    # 2. Question - what the user is asking
    # 3. Context - the retrieved documents
    # 
    # WHY "I don't know" fallback:
    # Prevents hallucinations - if context doesn't have the answer,
    # the LLM shouldn't make one up
    
    return prompt



In [4]:
# ============================================================================
# COMPONENT 3: SEARCH (Retrieval) - Placeholder for now
# ============================================================================

def search(question):
    """
    WHY THIS FUNCTION EXISTS:
    - This is the "Retrieval" part of RAG
    - Finds relevant documents from our knowledge base
    - For now, this is a placeholder - you'll implement real search later
    
    PARAMETERS:
    question (str): The user's question
    
    RETURNS:
    str: Retrieved context (documents)
    """
    
    # WHY: This is a placeholder
    # In lessons 4-5, you'll learn to:
    # 1. Download the FAQ dataset
    # 2. Index it with minsearch
    # 3. Actually search for relevant documents
    
    # For now, we manually return the answer we know is correct
    # This demonstrates how RAG works before we automate the search
    return """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit 
your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect 
to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning 
and submitting homework (while the form is open) without registering.
"""



In [5]:
# ============================================================================
# THE RAG PIPELINE (Orchestrates all 3 components)
# ============================================================================

def rag(question):
    """
    WHY THIS FUNCTION EXISTS:
    - This is the main RAG pipeline
    - Orchestrates all three components: Search → Prompt → LLM
    - The user calls this function with their question
    
    PARAMETERS:
    question (str): The user's question
    
    RETURNS:
    str: The final answer from the LLM
    """
    
    # STEP 1: RETRIEVAL
    # WHY: Search for relevant documents in our knowledge base
    # The better the search, the better the answer
    search_results = search(question)
    
    # STEP 2: BUILD PROMPT
    # WHY: Combine the question with retrieved context
    # This gives the LLM the information it needs to answer
    user_prompt = build_prompt(question, search_results)
    
    # STEP 3: GENERATION
    # WHY: Send the augmented prompt to the LLM
    # The LLM generates an answer based on the context we provided
    answer = llm(user_prompt)
    
    return answer

